In [ ]:
!pip install -q \
    sentence-transformers==2.7.0 \
    transformers==4.40.0 \
    langchain==0.1.20 \
    langchain-community==0.0.38 \
    langchain-core==0.1.52 \
    faiss-cpu \
    torch==2.1.0

import importlib, sys

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["torch", "sentence", "transformers", "langchain"]):
        del sys.modules[mod]


ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0)
ERROR: No matching distribution found for torch==2.1.0


In [ ]:
import os, json, warnings
import pandas as pd
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from google.colab import drive

warnings.filterwarnings("ignore")

CHUNKS_PATH  = "/content/chunks_temiz.jsonl"
FAISS_OUTPUT = "/content/drive/MyDrive/endodonti_rag/outputs/faiss"

os.makedirs(FAISS_OUTPUT, exist_ok=True)
print("Sabitler tanimlandı.")
print("CHUNKS_PATH :", CHUNKS_PATH)
print("FAISS_OUTPUT:", FAISS_OUTPUT)

Sabitler tanimlandı.
CHUNKS_PATH : /content/chunks_temiz.jsonl
FAISS_OUTPUT: /content/drive/MyDrive/endodonti_rag/outputs/faiss


In [ ]:
import os
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Drive zaten bagli.")

if os.path.exists(CHUNKS_PATH):
    print(f"OK: chunks_temiz.jsonl bulundu")
else:
    raise FileNotFoundError(f"BULUNAMADI: {CHUNKS_PATH}")

Drive zaten bagli.
OK: chunks_temiz.jsonl bulundu


In [ ]:
rows = []
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

docs = [
    Document(
        page_content=row["text"],
        metadata={"page": int(row["page"]), "source": "pathways.pdf"}
    )
    for row in rows
]

print(f"Toplam doc: {len(docs)}")
print(f"Ornek doc : {docs[0].page_content[:200]}")
print(f"Metadata  : {docs[0].metadata}")

Toplam doc: 2046
Ornek doc : Introduction Louis H. Berman, Kenneth M. Hargreaves The foundation of the specialty of endodontics is a gift from the generations of great endodontists and researchers before us. They guided us with t
Metadata  : {'page': 29, 'source': 'pathways.pdf'}


In [ ]:
MODELLER = {
    "MiniLM":     "sentence-transformers/all-MiniLM-L6-v2",
    "MPNet":      "sentence-transformers/all-mpnet-base-v2",
    "MultiQA":    "sentence-transformers/multi-qa-mpnet-base-cos-v1",
    "BGE_Base":   "BAAI/bge-base-en-v1.5",
    "E5_Base":    "intfloat/e5-base-v2",
    "PubMedBERT": "NeuML/pubmedbert-base-embeddings",
    "BGE_Large":  "BAAI/bge-large-en-v1.5"
}
for name, model_name in MODELLER.items():
    save_dir = os.path.join(FAISS_OUTPUT, name)
    if os.path.exists(os.path.join(save_dir, "index.faiss")):
        print(f"ATLA (zaten var): {name}")
        continue

    print(f"\n{'='*40}")
    print(f"Model: {name}")
    print(f"{'='*40}")

    try:
        emb = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True}
        )
        print("Embedding modeli yuklendi.")

        db = FAISS.from_documents(docs, emb)
        print("FAISS index olusturuldu.")

        os.makedirs(save_dir, exist_ok=True)
        db.save_local(save_dir)
        print(f"Kaydedildi: {save_dir}")

    except Exception as e:
        print(f"HATA — {name}: {e}")
        import traceback
        traceback.print_exc()


Model: MiniLM


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding modeli yuklendi.
FAISS index olusturuldu.
Kaydedildi: /content/drive/MyDrive/endodonti_rag/outputs/faiss/BGE_Large


In [ ]:
print("Mevcut FAISS klasörleri:")
print(f"{'Model':<12} {'Durum':<8} {'Dosyalar'}")
print("-" * 50)

for klasor in sorted(os.listdir(FAISS_OUTPUT)):
    tam_yol  = os.path.join(FAISS_OUTPUT, klasor)
    if not os.path.isdir(tam_yol):
        continue
    dosyalar  = os.listdir(tam_yol)
    faiss_var = "index.faiss" in dosyalar
    pkl_var   = "index.pkl"   in dosyalar
    durum     = "OK" if faiss_var and pkl_var else "EKSIK"
    print(f"{klasor:<12} {durum:<8} {dosyalar}")

Mevcut FAISS klasörleri:
Model        Durum    Dosyalar
--------------------------------------------------
BGE_Base     OK       ['index.faiss', 'index.pkl']
BGE_Large    OK       ['index.faiss', 'index.pkl']
E5_Base      OK       ['index.faiss', 'index.pkl']
MPNet        OK       ['index.faiss', 'index.pkl']
MiniLM       OK       ['index.faiss', 'index.pkl']
MultiQA      OK       ['index.faiss', 'index.pkl']
PubMedBERT   OK       ['index.faiss', 'index.pkl']
